# 1. Business Understanding

Primeira fase do CRISP-DM. Antes de mexer em qualquer dado, o objetivo aqui é entender o problema de negócio: o que a empresa quer resolver, por que isso importa, e quais perguntas específicas o trabalho vai tentar responder. É praticamente todo texto, com alguma tabela de apoio. Código fica para a partir do notebook 02.

## 1.1 Cenário e dor de negócio

### O contexto

A empresa do estudo é um e-commerce que cresceu rápido nos últimos anos. Mais pedidos, mais entregas, mais interações com atendimento. O crescimento trouxe ganhos de escala, mas também uma consequência menos óbvia: a experiência do cliente passou a variar muito de pessoa para pessoa. Dois clientes que, no papel, viveram a mesma jornada (mesmo tipo de pedido, mesma região, mesmo prazo de entrega) acabam dando notas de NPS bem diferentes. Um vira promotor, o outro vira detrator, e a empresa não consegue explicar o porquê só olhando para os indicadores operacionais que já coleta.

Esse é o cenário que define o problema de negócio.

### O NPS como indicador atrasado

O NPS hoje só é coletado depois que a jornada de compra acabou. Isso significa que a empresa só descobre que tem um detrator quando o cliente já está detratando. É um padrão clássico em monitoramento de qualquer sistema: você só vê o problema depois que ele virou consequência. Em engenharia de plataforma, a gente chama isso de *lagging indicator*, ou seja, uma métrica que confirma um evento que já aconteceu, em vez de antecipar um que está prestes a acontecer.

Uma referência interessante que encontrei sobre essa distinção é o conceito de *Balanced Scorecard*, do Kaplan e Norton, que separa indicadores em "lagging" (resultados) e "leading" (direcionadores). O verbete da Wikipedia tem um resumo bom para quem quer entender como métricas de negócio são organizadas: [Balanced Scorecard, Wikipedia](https://en.wikipedia.org/wiki/Balanced_scorecard).

O custo de operar só com indicador atrasado é alto. A empresa fica em modo reativo permanente. Quando o detrator finalmente aparece na pesquisa, três coisas costumam ter acontecido: ele já decidiu não comprar de novo, ele já pode ter falado mal da marca para amigos e família, e o problema operacional que o detratou provavelmente já afetou outros clientes parecidos com ele. Conseguir prever o NPS antes da pesquisa é, na prática, transformar um indicador atrasado em um indicador antecipado. É a diferença entre "saber que perdemos" e "agir antes de perder".

### Pontos de dúvidas não explicados

O ponto que torna esse problema interessante de Ciência de Dados é o seguinte: olhando só para os indicadores operacionais que a empresa coleta (tempo de entrega, contatos com atendimento, valor do pedido, região), não dá para separar bem promotores de detratores. Existem clientes com histórico operacional praticamente idêntico que avaliaram a empresa de forma oposta.

Isso pode estar acontecendo por três motivos diferentes, e cada um tem implicação diferente para o trabalho.

O primeiro é o de **variáveis omitidas**: existe algo que afeta a satisfação e que a empresa simplesmente não está medindo, como qualidade percebida do produto, tom da comunicação no SAC, ou expectativa formada antes da compra. O segundo é o de **interações entre variáveis**, em que o impacto de uma variável depende de outra. Três dias de atraso pode ser tolerável quando o cliente comprou um sofá, mas pode ser crítico quando comprou um remédio. Sozinha, a variável `delivery_delay_days` não conta a história inteira. O terceiro é o de **pontos de ruptura**: existe um limite acima do qual o cliente "vira a chave" e a satisfação cai de forma abrupta. Pequenas variações abaixo desse limite quase não mexem no NPS, mas pequenas variações acima dele derrubam a nota.

Esse último ponto, em especial, tem bastante respaldo na literatura de comportamento do consumidor. Uma referência que achei útil sobre o caráter não-linear da satisfação é o [Modelo Kano (Wikipedia)](https://en.wikipedia.org/wiki/Kano_model), que separa atributos de produto em três tipos (básicos, de performance e atrativos) e mostra como cada tipo gera satisfação de forma muito diferente.

As três explicações não são mutuamente exclusivas. É provável que o caso real envolva os três efeitos misturados. A tarefa do projeto é desemaranhar isso com os dados que existem.

### Por que isso machuca a aprte financeira da empresa

A dor não é só conceitual, ela tem impacto financeiro direto. Vejo pelo menos três vetores claros.

O primeiro é a **receita perdida por não-recompra**. Cada detrator é um cliente que provavelmente não vai voltar. O custo de aquisição (CAC) já foi pago, mas o retorno (LTV) é interrompido cedo demais.

O segundo é o **CAC inflado pelo boca-a-boca negativo**. Detrator costuma não silenciar; ele compartilha a má experiência em canais públicos, o que reduz a conversão de novos clientes e força a empresa a investir mais em marketing pago para compensar.

O terceiro é o **custo operacional do atendimento**. Detrator gera mais tickets, mais reclamações formais, mais devoluções, e cada um desses eventos tem um custo unitário que escala junto com a base de detratores.

Vale notar que dois desses três vetores estão observáveis no próprio dataset. A recompra em 30 dias (`repeat_purchase_30d`) é um proxy do primeiro vetor, e os contatos com atendimento e reclamações (`customer_service_contacts`, `complaints_count`) são proxies do terceiro. Isso é importante porque significa que conseguimos quantificar parte do custo da detração com os dados internos que já temos. O problema é mensurável dentro de casa, mesmo sem dados externos.

### Síntese

Resumindo o que temos: a empresa cresceu rápido, perdeu o padrão de satisfação dos clientes, ainda mede esse padrão de forma reativa, e gostaria de antecipar o NPS para agir antes que o cliente vá embora.

A pergunta-mãe que organiza todo o trabalho é:

> **Quais fatores operacionais explicam a variação do NPS, e é possível prever a nota antes da pesquisa, a partir de dados que a empresa já coleta?**

As próximas seções vão refinar essa pergunta em sub-perguntas mais específicas, mapear quem dentro da empresa se beneficia da resposta, e definir as hipóteses que vamos testar com os dados.

## 1.2 Por que NPS, e não outra métrica?

### A pergunta que parece óbvia mas não é

O enunciado define o NPS como métrica principal do trabalho. Mas vale parar e perguntar: por que NPS? O dataset já tem outras candidatas óbvias para representar satisfação, como o `csat_internal_score` (score interno calculado pela própria empresa), o `repeat_purchase_30d` (se o cliente voltou a comprar) e o `complaints_count` (quantas vezes ele reclamou formalmente). Se essas métricas existem e são mais fáceis de coletar, por que rodar uma pesquisa de NPS por cima?

A resposta está no que cada métrica realmente mede. Elas não são versões diferentes da mesma coisa, são coisas diferentes.

### O que cada métrica captura

A tabela abaixo separa as quatro métricas pelo tipo de informação que cada uma traz, com o horizonte de tempo em que ela faz sentido.

| Métrica | O que mede | Horizonte | Tipo de sinal |
|---|---|---|---|
| `csat_internal_score` | Satisfação calculada **pela empresa** com base em regras internas | Por pedido (transacional) | Score derivado, visão da empresa sobre o cliente |
| `complaints_count` | Insatisfação **manifesta** (cliente que reclamou formalmente) | Curto prazo | Comportamento autosselecionado (só capta quem quis se manifestar) |
| `repeat_purchase_30d` | **Comportamento revelado** numa janela curta | Curto prazo | Ação observada, não declarada |
| `nps_score` | **Intenção declarada** de recomendar a marca | Médio e longo prazo | Atitude reportada pelo próprio cliente |

A diferença mais importante entre elas é a separação entre **eventos** e **vínculo**. CSAT, recompra e reclamação são eventos; cada um deles olha para um momento específico da jornada. NPS é uma síntese atitudinal: é a forma como o cliente, depois de tudo, resume a relação dele com a marca em uma nota só. Por isso o NPS aproxima melhor o que importa de fato no longo prazo, que é a probabilidade do cliente continuar dando dinheiro para a empresa nos próximos meses e anos.

### O detrator silencioso

Tem um conceito que ajuda muito a entender por que o NPS é mais útil do que parece: o **detrator silencioso**. É o cliente que ficou insatisfeito, mas não reclamou. Ele simplesmente vai embora. Para a empresa, esse cliente é invisível em quase todas as métricas internas. Ele não aparece em `complaints_count` (não reclamou), provavelmente não vai aparecer em `repeat_purchase_30d` (não vai voltar), e o `csat_internal_score` dele pode até estar bom, porque a operação rodou normal do ponto de vista da empresa.

Existe uma estatística antiga e bastante repetida na área: a maioria dos clientes insatisfeitos não reclama, simplesmente para de comprar. Achei uma compilação de números atualizados sobre isso na [HelpScout, 75 Customer Service Facts](https://www.helpscout.com/75-customer-service-facts-quotes-statistics/), e a [SuperOffice tem outra lista parecida](https://www.superoffice.com/blog/customer-service-statistics/), com referências e fontes. Os números variam por estudo, mas o padrão é consistente: a reclamação formal capta apenas a ponta do iceberg da insatisfação.

A vantagem do NPS é justamente capturar parte desse silencioso, porque ele pergunta ativamente, em vez de esperar o cliente se manifestar. A pesquisa de NPS é uma amostra solicitada, não uma amostra autosselecionada como `complaints_count`. Isso reduz o viés de seleção (não elimina, porque quem responde a pesquisa também é uma autosseleção, só que menos extrema).

### A diferença entre satisfação e recomendação

Tem ainda uma distinção mais sutil, que aparece no próprio jeito como o NPS é perguntado. A pergunta canônica é "*de 0 a 10, quanto você recomendaria nossa empresa a um amigo ou colega?*". Note que ela não pergunta se o cliente está satisfeito. Ela pergunta se o cliente colocaria a reputação dele em jogo recomendando a empresa.

Essa diferença não é cosmética. Recomendar envolve um custo social que estar satisfeito não envolve. O cliente pode estar satisfeito com o pedido (CSAT alto) e ainda assim não recomendar (NPS baixo), porque achou a entrega lenta para o padrão que ele acha que um amigo esperaria, ou porque achou a comunicação fria. NPS mede o vínculo, não só o resultado da última compra.

A lógica original está no artigo [Frederick Reichheld, *The One Number You Need to Grow*, Harvard Business Review, 2003](https://hbr.org/2003/12/the-one-number-you-need-to-grow), que é o paper que cunhou o NPS. A consultoria que originou a métrica, Bain, mantém uma página explicando o conceito completo em [Introducing the Net Promoter System](https://www.bain.com/insights/introducing-the-net-promoter-system-loyalty-insights/), que é uma leitura mais leve para entender a ideia.

### CSAT interno vs. NPS: o blind spot operacional

Como o `csat_internal_score` aparece no dataset, vale comentar especificamente a relação dele com o NPS. O CSAT interno provavelmente é calculado pela empresa a partir de regras operacionais (entrega no prazo somou pontos, atendimento rápido somou pontos, atraso tirou pontos). Se essa hipótese estiver certa, o `csat_internal_score` é, no fundo, **a visão da empresa sobre o cliente**. Já o `nps_score` é **a visão do cliente sobre a empresa**.

Quando os dois divergem (CSAT interno alto e NPS baixo), a empresa está cega para alguma coisa. Pode ser uma variável que ela não está medindo, pode ser uma expectativa do cliente que a operação não capta, pode ser um problema de comunicação. Esse gap entre as duas métricas vai ser uma das hipóteses interessantes de investigar na EDA.

Existe também uma implicação técnica importante para a fase de modelagem: usar `csat_internal_score` como variável de entrada para prever NPS é arriscado, porque os dois podem ter sido calculados a partir das mesmas variáveis operacionais. Isso pode parecer uma feature poderosíssima, mas na prática pode ser uma forma de *target leakage* parcial. Vamos ter que tratar disso explicitamente quando chegarmos no notebook 03.

### As fragilidades do NPS

Passei a seção inteira defendendo o NPS, mas é importante dizer que ele tem problemas sérios. Se fosse perfeito, todo mundo usaria só ele, e não é o caso.

O primeiro ponto fraco é a **taxa de resposta baixa**. Pesquisa de NPS costuma ter retorno entre 5% e 20%, dependendo do canal, e quem responde é uma turma bem específica: clientes muito satisfeitos que querem elogiar, ou clientes muito insatisfeitos que querem reclamar. O cliente "morno", que provavelmente é a maioria, raramente abre o e-mail. O resultado é que o NPS observado tende a ser mais polarizado do que a realidade.

O segundo é que **NPS é uma nota, não um diagnóstico**. Saber que uma região tem NPS médio de 5,8 não diz por quê. Pode ser entrega, produto, preço, atendimento. O NPS sozinho serve como termômetro, não como bússola. Para virar bússola, ele precisa ser combinado com modelagem e EDA, que é exatamente o que vamos fazer aqui.

O terceiro é a **categorização em três grupos**, que tem um lado arbitrário. A regra original define detrator de 0 a 6, neutro 7 e 8, promotor 9 e 10. Pense no caso: um cliente que deu nota 6 e um que deu nota 0 são tratados como o mesmo grupo, o que joga muita variância fora. Há autores que questionam essa escolha há anos. Uma análise crítica acessível é a do [Reforge, NPS explained (and where it falls short)](https://www.reforge.com/blog/nps-net-promoter-score-explained), que comenta o que o NPS captura e o que ele perde.

Para este projeto, isso vira uma decisão prática: vamos rodar **regressão** (mantendo a escala 0 a 10, sem perder informação) **e classificação** (usando os buckets canônicos detrator/neutro/promotor), e comparar as duas. Cada abordagem responde uma pergunta diferente, e a escolha de qual usar na prática depende do que a empresa quer fazer com o resultado: priorização granular precisa de regressão, alarme rápido precisa de classificação.

### Resumo

O NPS é a melhor métrica disponível para medir vínculo do cliente com a marca, mas é uma métrica imperfeita que precisa ser entendida em conjunto com as outras (recompra, reclamação, CSAT) para fazer sentido completo. O trabalho aqui não é só prever NPS, é entender o que está por trás dele.

## 1.3 Stakeholders e áreas beneficiadas

### Os três papéis

Olhando para o problema da empresa, dá para separar quem se beneficia em três grupos distintos. Cada papel tem uma motivação diferente, um nível de senioridade diferente, e um tipo de ação diferente.

| Papel | Quem normalmente ocupa | Motivação principal | O que precisa do projeto |
|---|---|---|---|
| **Sponsor** (paga e decide) | CX / Customer Experience, ou direto C-Level (CEO, COO) | NPS é frequentemente um KPI de board. O sponsor quer subir o número agregado e mostrar isso no resultado da empresa | Visão estratégica do problema, magnitude do ganho potencial, justificativa de investimento |
| **Executor** (corrige na operação) | Logística (#1), SAC (#2), Pricing/Comercial (#3) | Cada área é a "dona" de algumas variáveis operacionais e tem o poder de mexer nelas | Ação concreta por cliente ou por segmento, integrável no fluxo de trabalho que já existe |
| **Cliente operacional** (consome insight no dia-a-dia) | SAC, Logística, Comercial, eventualmente Produto | Precisa priorizar tickets, rotas, contatos proativos, ofertas | Score por cliente ou por pedido, com explicação simples ("por que esse cliente foi sinalizado") |

A confusão clássica em projetos como esse é tratar todo mundo como se fosse o mesmo público. O slide executivo que vou apresentar para o CEO não é o mesmo dashboard que o supervisor de SAC vai abrir todo dia. O modelo é o mesmo, a entrega muda.

### Mapa de variáveis por área

Para deixar concreto onde cada área pode atuar, montei a tabela abaixo cruzando as variáveis do dataset com a área que tem mais influência sobre cada uma e que tipo de ação ela poderia tomar.

| Área | Variáveis do dataset que controla | Ações possíveis com base no modelo |
|---|---|---|
| **Logística** | `delivery_time_days`, `delivery_delay_days`, `freight_value`, `delivery_attempts` | Priorizar entregas de pedidos com risco alto de detração; renegociar SLA com transportadora em regiões críticas; redesenhar rota; abrir CD regional |
| **SAC / Atendimento** | `customer_service_contacts`, `resolution_time_days`, `complaints_count` | Priorizar fila de tickets pelo risco de detração; oferecer gesto de recuperação proativo (cupom, ligação) para clientes em risco; reescrever script para casos sensíveis |
| **Pricing / Comercial** | `order_value`, `discount_value`, `payment_installments`, `freight_value` | Ajustar política de frete por região; oferecer parcelamento estendido para clientes em risco; calibrar uso de desconto como retenção |
| **Produto** | (nenhuma direta no dataset) | Investigar categorias de produto associadas a maior detração (depende de enriquecer o dataset com categoria no futuro) |
| **CX / Estratégia** | (todas, indiretamente) | Direcionar orçamento entre as outras áreas conforme o impacto estimado de cada alavanca |

Note que **Produto não tem variável direta no dataset atual**. É uma limitação importante. Se a empresa quisesse de fato envolver Produto na análise, precisaria adicionar pelo menos categoria do produto comprado, que hoje não existe. Vou tratar isso como uma das limitações declaradas no fim do trabalho.

### Onde está o maior poder de ação

Entre as três áreas executoras, a logística é claramente a que tem maior poder de ação direto sobre o NPS. As variáveis logísticas (`delivery_time_days`, `delivery_delay_days`, `delivery_attempts`) são, intuitivamente, fortes candidatas a aparecerem como mais relevantes no modelo, e são variáveis que a empresa controla operacionalmente, com investimento e contrato de transportadora.

O SAC vem em segundo lugar. As variáveis dele (`customer_service_contacts`, `resolution_time_days`, `complaints_count`) são importantes, mas tem uma sutileza: parte delas é mais sintoma do que causa. O cliente que entra em contato com o SAC já costuma estar com algum problema; o tempo de resolução é o que pode ainda salvar a relação. Isso significa que a alavanca do SAC não é "evitar contatos", e sim "responder bem aos contatos que aconteceram". É uma alavanca real, mas mais limitada que a de logística, porque ela depende de a falha já ter começado.

Pricing e Comercial vêm em terceiro. As variáveis deles têm efeito indireto (`discount_value`, `payment_installments`), funcionando muito mais como modulação ou compensação do que como driver primário do NPS.

Essa hierarquia é uma hipótese a ser testada na fase 4 (EDA) e fase 5 (modelo). Se as variáveis logísticas aparecerem com maior importância no modelo, a hipótese se confirma e a recomendação fica fácil. Se aparecer um padrão diferente, melhor ainda, porque vai gerar um insight contraintuitivo, que costuma ser o tipo de resultado mais valioso para o negócio.

### Onde está o sponsor financeiro

Aqui vai uma distinção que vale para qualquer projeto de Ciência de Dados em empresa, e que muita gente confunde.

Quem **sente a dor** do NPS baixo no dia-a-dia é o SAC. É o time que recebe a ligação do cliente irritado, o ticket no sistema, a reclamação no Reclame Aqui. O supervisor de SAC sabe melhor do que ninguém que tem clientes detratando. Faz sentido pensar em SAC como sponsor por essa razão.

Mas quem **paga a conta** de um projeto desses não costuma ser o SAC. SAC, como área operacional, raramente tem orçamento para contratar projeto de DS, redesenhar dashboards e investir em modelo preditivo. Quem libera esse tipo de orçamento normalmente é a área de Customer Experience (CX), em estruturas mais maduras, ou direto o C-Level (CEO, COO), em estruturas mais enxutas. Esse é o sponsor financeiro real.

A diferença prática é importante. O SAC é **cliente** do projeto (consome o resultado), não sponsor. O slide executivo precisa ser desenhado para o sponsor real (CX/C-Level), com a linguagem que esse público entende: impacto em LTV, redução de churn, retorno sobre o investimento no projeto. Apresentar esse mesmo slide para um supervisor de SAC seria desperdiçar oportunidade, porque o supervisor não tem alavanca para autorizar investimento, ele tem alavanca para usar o que for entregue.

### Resumo

Logística é quem mais pode mexer no NPS. SAC sente a dor mas tem alavanca limitada. CX ou C-Level é quem paga a conta. Cada um precisa de um material diferente.

## 1.4 Como o NPS impacta o negócio

NPS sozinho é só um número. Ele só faz diferença para a empresa quando está conectado a comportamentos que afetam o caixa. Esta seção explica por que NPS alto vira mais receita, por que NPS baixo vira mais custo, e por que o crescimento da empresa pode estar quebrando esse loop justamente agora.

### Recompra: por que promotor volta e detrator não

A primeira ponte entre NPS e dinheiro é a recompra. Promotor tende a comprar de novo, detrator tende a sumir. A pergunta interessante não é se isso acontece, e sim por quê.

Consigo enxergar pelo menos quatro mecanismos diferentes operando ao mesmo tempo.

O primeiro é o **custo de troca**. Recomeçar com outro fornecedor exige tempo, cadastro novo, descobrir prazo de entrega, aprender a usar o app. Promotor não vê motivo para pagar esse custo; detrator vê qualquer motivo como suficiente.

O segundo é o **viés de confirmação**. Cliente satisfeito interpreta uma experiência neutra como positiva ("foi normal, devo voltar"); detrator interpreta a mesma experiência neutra como negativa ("já tinha desconfiado").

O terceiro é o **hábito**. A recompra de quem já comprou e ficou satisfeito é uma decisão de baixo esforço cognitivo, quase automática. O detrator quebra essa inércia e passa a procurar concorrente ativamente, o que demanda esforço, mas o esforço é justamente o sinal de que ele está disposto a sair.

O quarto é a **percepção de risco**. Comprar online sempre tem risco percebido (será que o produto chega? será que vem certo? será que tem trabalho de troca?). Promotor já validou que esse risco é baixo nessa empresa específica; detrator validou o oposto e generaliza a desconfiança para qualquer compra futura.

A operacionalização disso no nosso dataset é a variável `repeat_purchase_30d`. É uma janela curta, mas serve como proxy. Uma das hipóteses que vamos testar mais à frente é exatamente quanto da variação em `repeat_purchase_30d` o `nps_score` consegue explicar. Se NPS prediz recompra de forma significativa, o valor econômico do projeto fica claro. Se não prediz, todo o argumento de impacto financeiro precisa ser revisto.

A referência clássica sobre o impacto de retenção em e-commerce é o paper do Reichheld de 1990, [Zero Defections: Quality Comes to Services (HBR, 1990)](https://hbr.org/1990/09/zero-defections-quality-comes-to-services), que popularizou a famosa regra de que aumentar a retenção em 5% pode aumentar o lucro entre 25% e 95%. É uma estimativa antiga e específica do contexto dele, mas a lógica continua sendo a base de quase toda discussão sobre LTV. Para uma versão mais recente e digerível, achei útil o artigo da HBR de 2014, [The Value of Keeping the Right Customers](https://hbr.org/2014/10/the-value-of-keeping-the-right-customers). E para uma definição formal de CLV, o [verbete Customer Lifetime Value na Wikipedia](https://en.wikipedia.org/wiki/Customer_lifetime_value) cobre bem.

### Boca-a-boca: a assimetria entre detrator e promotor

A segunda ponte entre NPS e dinheiro é o boca-a-boca, que afeta o quanto a empresa precisa gastar para adquirir cliente novo. Aqui mora um dos pontos mais interessantes do NPS: detrator e promotor não se comportam de forma simétrica.

Detrator é mais **vocal em canais públicos**. Posta no Reclame Aqui, marca a empresa em redes sociais, deixa review com nota baixa, conta para o grupo da família no WhatsApp. A motivação por trás disso é estudada na psicologia como **viés de negatividade**: humanos têm tendência evolutiva a compartilhar e prestar mais atenção em informação negativa do que em positiva, porque historicamente saber o que é perigoso teve mais valor de sobrevivência do que celebrar o que é bom. Uma referência conceitual sobre isso é o [verbete Negativity bias na Wikipedia](https://en.wikipedia.org/wiki/Negativity_bias).

Promotor é mais **influente em canais privados**. Ele recomenda a empresa quando alguém pergunta diretamente ("onde você comprou esse?", "qual loja confiável?"), em conversa íntima, em momento de decisão de compra do amigo. É um boca-a-boca menos público, mas com taxa de conversão maior, porque vem de fonte de alta confiança. O [verbete Word of mouth na Wikipedia](https://en.wikipedia.org/wiki/Word_of_mouth) sintetiza bem essa diferença.

Essa assimetria tem implicação direta para o argumento de negócio. Detratores destroem **consideração pública** no curto prazo (um cliente prestes a comprar vê reviews ruins e desiste). Promotores constroem **consideração íntima** no longo prazo (cliente novo aparece via indicação de amigo). Não é que um vale mais que o outro: eles operam em horizontes e canais diferentes.

A consequência prática é que **prevenir detração tende a ter retorno mais rápido do que cultivar promoção**. Não porque promoção valha menos, mas porque o dano público de um detrator se materializa quase imediato em conversão perdida, enquanto o benefício de um promotor se distribui ao longo de meses ou anos via recomendações pontuais.

Não temos dados de boca-a-boca diretamente no dataset, mas o argumento serve para o trabalho: cada detrator antecipado é não só um cliente preservado, é também uma série de aquisições futuras que não foram perdidas por reputação pública corroída.

### Market share: o loop que se autoalimenta (e quebra)

A terceira ponte entre NPS e dinheiro é a mais estratégica. NPS afeta market share por meio de um loop econômico que se autoalimenta enquanto está saudável, e se autodestrói quando quebra.

O loop saudável funciona assim: NPS alto reduz churn, fazendo cada cliente valer mais no tempo, e ao mesmo tempo gera boca-a-boca positivo, que baixa o custo de aquisição efetivo. Com LTV maior e CAC menor, a empresa tem margem extra para reinvestir em aquisição paga e em melhoria de produto, o que aumenta volume, e volume bem operacionalizado mantém NPS alto, fechando o ciclo. O [verbete Loyalty business model na Wikipedia](https://en.wikipedia.org/wiki/Loyalty_business_model) descreve essa lógica em mais detalhe.

O problema é que esse loop tem um ponto de quebra. Quando a empresa cresce em volume mais rápido do que consegue manter qualidade operacional, o NPS começa a cair. Cai porque entrega vira inconstante, porque SAC fica sobrecarregado, porque a base de clientes fica heterogênea demais para ser atendida com o mesmo padrão. A partir desse momento, todo o loop se inverte: NPS cai, recompra cai, LTV cai, boca-a-boca vira negativo, CAC sobe, a margem extra desaparece, e o crescimento fica caro demais para sustentar.

É exatamente esse o cenário que o enunciado descreve. A empresa do estudo cresceu rápido, ganhou escala, mas perdeu o padrão de satisfação. O loop está justamente nesse ponto de quebra. Cada novo detrator que a escala gera neutraliza a vantagem econômica de vários promotores conquistados nos canais pagos. O projeto não é apenas sobre "prever NPS", é sobre antecipar onde a perda de qualidade operacional está corroendo o ganho do crescimento.

### Resumo

NPS alto puxa recompra (LTV sobe). NPS alto puxa boca-a-boca positivo (CAC cai). A combinação dos dois move market share. Mas o loop se quebra quando o crescimento operacional vira mais rápido do que a capacidade de manter qualidade, e é exatamente onde a empresa do nosso caso parece estar agora.

## 1.5 Indicadores de mercado que enriqueceriam a análise

O dataset que temos é razoavelmente completo para o que ele se propõe, mas só contém dados internos da empresa. Para o trabalho ficar fechado, vale refletir sobre que tipo de informação externa ajudaria a interpretar melhor os resultados, mesmo que não seja viável coletar tudo agora. Não vou ir atrás dessas fontes de fato; é exercício de raciocínio sobre o que está fora do alcance do dataset e por que faz diferença.

Organizei isso em três eixos: calibração, concorrência e contexto.

### Eixo 1: calibração de mercado

NPS isolado não diz nada. Um número como "NPS médio de 6,5" só vira informação útil quando comparado com alguma referência externa. Por isso a primeira família de dados externos relevantes é a de **benchmarks setoriais**.

Existem consultorias e plataformas que publicam médias por setor, mostrando quanto e-commerce em geral, e-commerce de cada categoria, varejo físico, etc. Uma fonte aberta com benchmarks razoavelmente atualizados é o [Retently NPS benchmarks](https://retently.com/blog/good-net-promoter-score/), que serve como ponto de partida. Saber que e-commerce em geral roda em torno de NPS 50 (numa escala de -100 a +100) muda completamente a leitura do número da nossa empresa, porque permite entender se ela está dentro do padrão setorial ou abaixo dele.

A mesma lógica vale para o tempo de entrega. "5 dias" só vira informação útil quando comparado com a mediana do setor: se o e-commerce brasileiro de massa entrega em torno de 4 dias úteis, 5 está acima da média e cada dia extra está corroendo NPS; se a mediana for 7, 5 dias é uma vantagem competitiva. Pesquisas como a Conversion ou a Ebit Nielsen publicam esses dados periodicamente para o mercado brasileiro.

### Eixo 2: concorrência e o efeito de ancoragem

A segunda dimensão é o que os concorrentes diretos fazem, e como isso forma a expectativa do cliente. A satisfação não é função do que a empresa prometeu; é função da diferença entre a expectativa do cliente e o que ele recebeu, e essa expectativa é formada por experiências externas, principalmente com o melhor concorrente que ele já viu no setor.

Esse fenômeno é conhecido em psicologia como **efeito de ancoragem**, descrito originalmente por Kahneman e Tversky. Uma referência leve sobre isso é o [verbete Anchoring effect na Wikipedia](https://en.wikipedia.org/wiki/Anchoring_effect). Para uma referência um pouco mais formal sobre essa dinâmica de expectativa versus entrega no contexto de qualidade de serviço, o [SERVQUAL na Wikipedia](https://en.wikipedia.org/wiki/SERVQUAL) é uma referência clássica que vale dar uma olhada.

Aplicado ao nosso caso, o impacto é grande. Mesmo que a empresa entregue exatamente no prazo prometido, o cliente pode ainda assim ser detrator se a referência mental dele é Amazon Prime, que entrega em 1 ou 2 dias. A âncora se moveu nos últimos anos, e toda empresa de e-commerce passou a ser comparada com ela, mesmo quando não compete no mesmo segmento. A implicação prática é que reduzir o `delivery_time_days` médio em um dia pode não subir o NPS se o mercado também acelerou. NPS é mais ou menos uma corrida em que a empresa precisa correr para ficar parada. Isso precisa estar nas limitações do projeto, porque é o tipo de fator que estraga conclusão otimista.

### Eixo 3: contexto (sazonalidade, regional, redes sociais)

A terceira dimensão é sobre fatores que não estão no dataset mas afetam a leitura do que está nele.

O primeiro é a **sazonalidade**. Eventos como Black Friday, Natal e Dia das Mães mudam volume, prazo de entrega e capacidade do SAC. NPS coletado em pico de Black Friday provavelmente é mais baixo do que em mês comum, sem que isso signifique que a empresa piorou estruturalmente. O nosso dataset não tem timestamp, então não conseguimos analisar sazonalidade nem controlar por ela. É uma das limitações mais relevantes do trabalho. O [verbete Black Friday na Wikipedia](https://en.wikipedia.org/wiki/Black_Friday_(shopping)) ajuda a contextualizar o tamanho desse efeito no comércio.

O segundo é o **contexto socioeconômico regional**. O mesmo atraso pode ser tolerado de forma diferente por clientes de regiões com perfis de renda e expectativa diferentes. Cliente de região com poder aquisitivo maior pode ter expectativa de serviço mais alta e tolerar menos falha. Cliente de região com acesso historicamente pior pode valorizar mais um pedido que chega no prazo. Saber renda média, nível de urbanização e densidade logística por região ajudaria a interpretar a variável `customer_region`. Os dados básicos para isso estão disponíveis no [IBGE](https://www.ibge.gov.br/) e poderiam ser cruzados com os clientes.

O terceiro é o **sentimento em redes sociais**. Hoje existem ferramentas de social listening que rastreiam menções de marca e classificam por sentimento. Picos de reclamação em redes costumam preceder quedas no NPS pesquisado, porque o canal social é mais imediato. Não temos esse dado e seria caro construir, mas é uma evolução natural de quem quiser ir além da pesquisa formal.

Uma quarta fonte que vale citar é a **reputação pública estruturada**, especialmente Reclame Aqui no contexto brasileiro. Score público da empresa, taxa de resposta, percentual de problemas resolvidos. Comparar a nota interna do `csat_internal_score` com a nota pública do Reclame Aqui poderia revelar mais um tipo de "blind spot" reputacional.

## 1.6 Pergunta de pesquisa e agenda de hipóteses

### Da pergunta de negócio à pergunta de pesquisa

A pergunta-mãe levantada na seção 1.1 era genérica de propósito: "quais fatores explicam a variação do NPS, e dá para prever a nota antes da pesquisa?". Para virar pesquisa testável, ela precisa ser quebrada em duas perguntas mais específicas.

A primeira é **descritiva**: dadas as variáveis operacionais que a empresa coleta (logística, atendimento, perfil do cliente e do pedido), quais delas têm relação com a variação no NPS, e em que magnitude? É a pergunta que a EDA vai responder.

A segunda é **preditiva**: usando essas mesmas variáveis como entrada, é possível construir um modelo que estime o NPS de um cliente antes da pesquisa, com qualidade boa o suficiente para apoiar decisão operacional? É a pergunta que a fase de modelagem vai responder.

A primeira pergunta busca interpretação (entender o que importa); a segunda busca uso prático (entregar uma estimativa). Um modelo pode ser bom em uma e ruim na outra. O trabalho vai tentar responder as duas, sabendo que o público do slide executivo se interessa mais pela primeira (por que clientes detratam) e o público de manutenção do modelo se interessa mais pela segunda (qual a precisão da estimativa).

### Sub-perguntas operacionais

Para guiar a EDA e a modelagem, separei a pergunta-mãe em quatro sub-perguntas, cada uma cobrindo uma área operacional:

- **Logística:** o tempo e a confiabilidade da entrega têm efeito linear sobre o NPS, ou existe um ponto de ruptura a partir do qual a satisfação cai de forma abrupta?
- **Atendimento:** o cliente que entra em contato com o SAC já é mais detrator por natureza, ou um atendimento bem resolvido pode reverter a insatisfação inicial?
- **Pedido e cliente:** o impacto de uma falha operacional depende do tipo de pedido (valor, parcelas, desconto) e do perfil do cliente (idade, região, tempo de relacionamento)?
- **Indicadores internos:** como o `csat_internal_score` se relaciona com o `nps_score`, e o que a divergência entre os dois revela sobre os pontos cegos da operação?

### Hipóteses em logística

| # | Hipótese | Variáveis | Onde testar | Técnica |
|---|---|---|---|---|
| **H1** | Existe um ponto de ruptura em `delivery_time_days` acima do qual o NPS cai de forma desproporcional ao aumento do tempo | `delivery_time_days`, `nps_score` | EDA (04) e modelo (05) | Comparação de médias de NPS entre faixas de `delivery_time_days` usando teste de hipóteses para diferença de médias; intervalo de confiança das médias por faixa. No modelo, regressão linear com a variável categorizada por faixas |
| **H2** | A tolerância a atraso de entrega depende do valor do pedido | `delivery_delay_days`, `order_value`, `nps_score` | Modelo (05) | Regressão linear múltipla com termo de interação `delivery_delay_days * order_value`; teste de hipótese sobre o coeficiente da interação |
| **H7** | A variação de NPS por região é explicada pelas diferenças operacionais (tempo de entrega, tentativas), não pela região em si | `customer_region`, `delivery_time_days`, `delivery_attempts`, `nps_score` | EDA (04) e modelo (05) | Comparação de médias de NPS por região com teste de hipóteses; depois, regressão linear incluindo a região e as variáveis operacionais para ver se o efeito da região diminui quando controlado pelas operacionais |

### Hipóteses em atendimento

| # | Hipótese | Variáveis | Onde testar | Técnica |
|---|---|---|---|---|
| **H3** | Cliente que reclamou e teve resolução rápida tem NPS comparável ao de quem não reclamou | `complaints_count`, `resolution_time_days`, `nps_score` | EDA (04) e modelo (05) | Separar três grupos (não reclamou, reclamou e foi resolvido rápido, reclamou e foi resolvido devagar) e comparar a média de NPS entre eles dois a dois com teste de hipóteses para diferença de médias e intervalo de confiança |
| **H6** | Cliente com zero contato no SAC não é necessariamente promotor; pode ser detrator silencioso | `customer_service_contacts`, `complaints_count`, `nps_score` | EDA (04) | Comparar a média e o intervalo de confiança do NPS entre o grupo com zero contatos e o grupo com pelo menos um contato; teste de hipóteses para diferença de médias |

### Hipóteses em indicadores internos

| # | Hipótese | Variáveis | Onde testar | Técnica |
|---|---|---|---|---|
| **H4** | `csat_internal_score` e `nps_score` divergem em certos perfis, mostrando blind spots da operação | `csat_internal_score`, `nps_score`, demais covariáveis | EDA (04) | Correlação de Pearson entre as duas variáveis; análise descritiva da diferença entre CSAT normalizado e NPS; segmentação dos casos com maior divergência |
| **H5** | NPS prediz `repeat_purchase_30d`: promotores recompram em proporção maior que detratores | `nps_score`, `repeat_purchase_30d` | EDA (04) e modelo separado | Cálculo de proporção de recompra dentro de cada faixa de NPS (detrator, neutro, promotor); teste de hipóteses para diferença de proporções entre detratores e promotores; intervalo de confiança para a diferença, com base na distribuição binomial |

### O que isso desbloqueia

Com a pergunta-mãe quebrada e as hipóteses listadas, a próxima fase do CRISP-DM (Data Understanding, no notebook 02) tem foco claro: confirmar se o dataset oferece a qualidade e a profundidade necessárias para testar essas sete hipóteses. Se alguma variável não tiver variabilidade suficiente, ou tiver missing demais, ou tiver distribuição inesperada, a hipótese correspondente precisará ser ajustada antes de chegar na EDA.

## 1.7 Limitações conhecidas

Antes de fechar a Fase 1, vale listar o que o trabalho não consegue fazer com os dados disponíveis. Declarar isso aqui evita que conclusões mais à frente sejam interpretadas além do que os dados sustentam.

### Dataset sem informação temporal

O CSV não tem timestamp dos pedidos. Sem essa coluna, não dá para analisar nem controlar pelo efeito de sazonalidade. NPS de Black Friday, Natal e Dia das Mães provavelmente é diferente do NPS de meses comuns, porque volume cresce, prazo de entrega aperta e SAC fica sobrecarregado. Como não temos data, qualquer média global pode estar misturando períodos muito diferentes, e não há como separar.

Vamos seguir tratando o dataset como se fosse uma fotografia atemporal, e isso vai ser declarado como aviso nos slides e no vídeo executivo. É uma das limitações mais importantes, porque sazonalidade tem efeito grande no comportamento do cliente.

### Risco com `csat_internal_score` como variável de entrada

O `csat_internal_score` é um score calculado pela própria empresa, e é provável que tenha sido construído a partir das mesmas variáveis operacionais que estão no dataset (entrega no prazo, atendimento rápido, etc.). Se essa hipótese estiver certa, usar essa variável como entrada do modelo pode dar uma falsa sensação de qualidade, porque ela carrega parte da informação do que a gente tenta prever.

Para tratar isso, vamos rodar o modelo de duas formas: uma com o `csat_internal_score` incluído, outra sem. A diferença de desempenho entre as duas vai mostrar quanto o score está dominando. Para uso prático, o modelo "honesto" provavelmente é o sem o `csat_internal_score`, porque na produção a empresa vai querer prever NPS a partir de variáveis cruas, não a partir de outro score derivado.

### Ausência de dados externos

Nenhum dado do mundo fora da empresa entra no trabalho. Não temos benchmarks de NPS por setor, tempo médio de entrega de mercado, dados de concorrência, dados socioeconômicos do IBGE, sentimento em redes sociais, ou nota pública no Reclame Aqui. Isso significa que qualquer comparação que a análise faça é interna: dentro da própria empresa, entre clientes, entre regiões, entre tipos de pedido.

A consequência é que conclusões do tipo "NPS de 6,5 é bom ou ruim" não vão ser respondidas de forma absoluta, só relativa. Para o escopo do trabalho isso é aceitável, mas é uma limitação que precisa estar clara.

### Categoria de produto não está no dataset

A área de Produto, que aparece no enunciado como uma das que se beneficiariam do projeto, não tem variável direta no dataset. Não sabemos se o cliente comprou um eletrônico, um item de moda, um produto de mercado, um remédio. Isso impede testar hipóteses do tipo "categoria de produto modula o efeito do atraso", que seria muito útil para Produto.

A recomendação no fim do trabalho vai ser que a empresa enriqueça o dataset com pelo menos uma coluna de categoria, em uma próxima rodada do projeto.

### Possível confusão entre causa e sintoma em algumas variáveis

Algumas variáveis do dataset podem confundir causa com efeito. O exemplo mais claro é `customer_service_contacts`: se o modelo mostrar que mais contatos com SAC estão associados a NPS mais baixo, isso provavelmente não significa que o contato em si machucou o cliente. O mais provável é que o contato seja sintoma de um problema que já existia (atraso, produto errado, dúvida não respondida), e o que machucou foi o problema, não o contato.

A consequência é que algumas correlações que vão aparecer na EDA precisam ser interpretadas com cuidado, e a recomendação de negócio derivada delas não pode ser "reduzir contatos com o SAC", porque isso seria atacar o sintoma e não a causa.